In [1]:
from sklearn.datasets import fetch_openml

from Lab3.MNIST_SGD import checkpoint

# Getting Data
data = fetch_openml(data_id=43939, as_frame=False, parser='auto')

data

{'data': array([[-122.23, 37.88, 41, ..., 126, 8.3252, 'NEAR BAY'],
        [-122.22, 37.86, 21, ..., 1138, 8.3014, 'NEAR BAY'],
        [-122.24, 37.85, 52, ..., 177, 7.2574, 'NEAR BAY'],
        ...,
        [-121.22, 39.43, 17, ..., 433, 1.7, 'INLAND'],
        [-121.32, 39.43, 18, ..., 349, 1.8672, 'INLAND'],
        [-121.24, 39.37, 16, ..., 530, 2.3886, 'INLAND']], dtype=object),
 'target': array([452600, 358500, 352100, ...,  92300,  84700,  89400]),
 'frame': None,
 'categories': {'ocean_proximity': ['<1H OCEAN',
   'NEAR BAY',
   'NEAR OCEAN',
   'INLAND',
   'ISLAND']},
 'feature_names': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'population',
  'households',
  'median_income',
  'ocean_proximity'],
 'target_names': ['median_house_value'],
 'DESCR': 'Median house prices for California districts derived from the 1990 census.\n\nDownloaded from openml.org.',
 'details': {'id': '43939',
  'name': 'california_housing',
  'version': 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np


# Feature Encoding - ocean proximity
ocean_proximity_map = {
    'ISLAND':     0,
    '<1H OCEAN':  1,
    'NEAR OCEAN': 2,
    'NEAR BAY':   3,
    'INLAND':     4,
}
mapper = np.vectorize(ocean_proximity_map.get)
data.data[:,8] = mapper(data.data[:,8])


X, y = data.data.astype(np.float32), data.target.astype(np.float32)

# Train-Val-Split 60/20/20
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42  # 0.25 * 0.8 = 0.2
)

# Data Normalization
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train)
X_val   = scaler_X.transform(X_val)
X_test  = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val   = scaler_y.transform(y_val.reshape(-1, 1)).ravel()
y_test  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Input_size: {X_train.shape[1]}")

Train: (12384, 9), Val: (4128, 9), Test: (4128, 9)
Input_size: 9


# MLP regressor

In [3]:
import os
from torch import nn

class MLPRegressor(nn.Module):
    def __init__(self, input_size:int, hidden_size:int, n_hidden:int,  output_size:int):
        super(MLPRegressor, self).__init__()

        self.input_layer = nn.Linear(input_size, hidden_size)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.relu(layer(x))

        x = self.output_layer(x)

        return x

os.makedirs('Models', exist_ok=True)

## Training function
Simple training loop with no Cross-validation

In [8]:
from matplotlib import pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim


class ModelTrainer:
    def __init__(self, X_train:np.ndarray, y_train:np.ndarray, X_val:np.ndarray, y_val:np.ndarray, batch_size:int, lr:float, n_epochs:int, dir:str = "Models"):
        self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.learning_rate = lr
        self.n_epochs = n_epochs

        self.X_train = X_train
        self.y_train = y_train

        self.X_val = X_val
        self.y_val = y_val

        self.dir = dir

    def _prepare_loader(self, X, y):
        #TODO: Given np.ndarrays transforms them into tensors and return loader
        X_tensor = torch.from_numpy(X).float().to(self.DEVICE)
        y_tensor = torch.from_numpy(y).float().to(self.DEVICE)
        data = TensorDataset(X_tensor, y_tensor)

        loader = DataLoader(data, batch_size=self.batch_size, shuffle=True)
        return loader

    def _train_one_epoch(self, model, loader, optimizer, criterion):
        DEVICE = self.DEVICE

        model.train()
        running_loss = 0.0
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X), y).backward()
            optimizer.step()
        avg_loss = running_loss / len(loader)

        return avg_loss

    def _calculate_val_loss(self, model, loader, criterion):
        DEVICE = self.DEVICE

        model.eval()
        running_loss = 0.0
        with torch.no_grad():
            for X,y in loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                output = model(X)
                loss = criterion(output, y)
                running_loss += loss.item()
        avg_loss = running_loss / len(loader)

        return avg_loss

    def _show_loss(self, history:dict, name):
        plt.figure(figsize=(8, 5))
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Val Loss')
        plt.title(f'Loss Curve - {name}')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()

    def train_network(self, models:dict):
        DEVICE = self.DEVICE


        checkpoints = {name:[] for name in models.keys()}
        for name, model in models.items():
            train_loader = self._prepare_loader(X_train, y_train)
            val_loader = self._prepare_loader(X_val, y_val)

            criterion = nn.MSELoss()
            optimizer = optim.Adam(model.parameters(), lr=self.learning_rate)


            best_val_loss = float("inf")
            history = {'train_loss': [], 'val_loss': []}

            model.to(self.DEVICE)
            for epoch in range(self.n_epochs):
                train_loss =self._train_one_epoch(model, train_loader, criterion)
                history['train_loss'].append(train_loss)

                val_loss = self._calculate_val_loss(model, val_loader, criterion)
                history['val_loss'].append(val_loss)

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    checkpoint = {
                        'epoch': epoch,
                        'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'best_val_loss': best_val_loss,
                        'history': history
                    }
                    PATH = os.path.join(self.dir, f'{name}.pth')
                    torch.save(checkpoint, PATH)
                    #print(f"Epoch {epoch}: Model Saved with new best loss {avg_val_loss:.4f}")

            self._show_loss(history, name)
            checkpoints[name] = checkpoint

        return checkpoints


trainer = ModelTrainer(X_train, y_train, X_val, y_val, batch_size=256, lr=0.001, n_epochs=100, dir="Models")
